In [ ]:
ip = "localhost"
port = 3001
end_point = "http://" + ip + ":" +str(port)
location = 'AMERICA'

In [ ]:
ip = "localhost"
port = 3002
end_point = "http://" + ip + ":" +str(port)
location = 'CHINA'

In [ ]:
#coding=utf-8
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP
from base64 import b64encode, b64decode
from Crypto.Hash import SHA,SHA256
from Crypto.Signature import PKCS1_v1_5
import json, os
import requests
import gzip
import zlib,binascii
private_key_path = str(port) + "/client_private_rsa_key.bin"
public_key_path = str(port) + "/client_rsa_public.pem"
working_dir = str(port) + "/data/"
ori_data = b''

def gz(filename):
    f_in = open(filename, 'rb')  
    f_out = gzip.open(filename + '.gz', 'wb')  
    f_out.writelines(f_in)  
    f_out.close()  
    f_in.close()  
    return filename + '.gz'

def get_publickey_from_server():
    r = requests.get(end_point + "/getPublicKey")
    public_key = bytes(r.json()['public_key'], encoding = 'utf8')
    return public_key

server_public_key = get_publickey_from_server()

def send_to_server(data):
    headers = {'content-type': 'application/json'}
    url = end_point + "/addData"
    params = {}
    return requests.post(url, params=params, data=json.dumps(data), headers=headers)

def CreateRSAKeys():
    #code = location + 'REMOVED-PASSPHRASE' + str(port)
    key = RSA.generate(2048)
    encrypted_key = key.exportKey(pkcs=8, protection="scryptAndAES128-CBC")
    with open(private_key_path, 'wb') as f:
        f.write(encrypted_key)
    with open(public_key_path, 'wb') as f:
        f.write(key.publickey().exportKey())

def calculate_hash(data_to_hash):
    hash_object = SHA256.new(data_to_hash)
    return hash_object.hexdigest()
        
#client side    
def encrypt_client(filename):         
    data = ''
    original_data = b''
    #fnamegz = filename
    fnamegz = gz(filename)
    with open(filename, 'rb') as f:
        original_data = f.read()
    with open(fnamegz, 'rb') as f:
        data = f.read()
    temp_file_name = fnamegz + '.b'
    with open(temp_file_name, 'wb') as out_file:
        recipient_key = RSA.import_key(server_public_key)
        session_key = get_random_bytes(16)
        # Encrypt the session key with the public RSA key
        cipher_rsa = PKCS1_OAEP.new(recipient_key)
        out_file.write(cipher_rsa.encrypt(session_key))
        # Encrypt the data with the AES session key
        cipher_aes = AES.new(session_key, AES.MODE_EAX)
        ciphertext, tag = cipher_aes.encrypt_and_digest(data)
        out_file.write(cipher_aes.nonce)
        out_file.write(tag)
        out_file.write(ciphertext)
    with open(temp_file_name, 'rb') as f:
        data = f.read()
    os.remove(temp_file_name)
    os.remove(fnamegz)
    return original_data,b64encode(data)

#client side
def generate_rsa_sha1_signature(data, private_key):
    hash_obj = SHA.new(data)
    key = RSA.importKey(private_key)
    signer = PKCS1_v1_5.new(key)
    signature = signer.sign(hash_obj)
    return b64encode(signature)

def clientSide(filename): 
    print('Working at client')
    original_data, encrypt_by_server_pub_base64_data = encrypt_client(filename)
    with open(private_key_path) as f:
        private_key = f.read()
    data_signature = generate_rsa_sha1_signature(encrypt_by_server_pub_base64_data, private_key)
    public_key = RSA.importKey(private_key).publickey().exportKey()
    return original_data,{'base64Data' : encrypt_by_server_pub_base64_data.decode('utf-8'), 'signature' : data_signature.decode('utf-8'), 'public_key' : public_key.decode('utf-8')}


def main():
    for (dirpath, dirnames, filenames) in os.walk(working_dir):
        for filename in filenames:
            print(filename)
            #print(dirpath + filename)
            original_data , send_to_server_data = clientSide(dirpath + filename)
            #print(original_data)
            """
            for data_key in send_to_server_data:
                print("------"+data_key+"------not to b64-------------------------")
                print("before:" + str(len(send_to_server_data[data_key])))
                send_to_server_data[data_key] = binascii.hexlify(zlib.compress(bytes(send_to_server_data[data_key],encoding = "utf8"),zlib.Z_BEST_COMPRESSION))
                print("After:" + str(len(send_to_server_data[data_key])))
                print("-------"+data_key+"---------to b64-------------------------")
                print("before:" + str(len(b64encode(send_to_server_data[data_key]))))
                send_to_server_data[data_key] = zlib.compress(b64encode(send_to_server_data[data_key]),zlib.Z_BEST_COMPRESSION)
                print("After:" + str(len(send_to_server_data[data_key])))
                print("-----------------------------------------------------")
            """
            send_to_server_data['certificateOwner'] = filename
            send_to_server_data['fileHash'] = calculate_hash(original_data)
            #print(send_to_server_data)
            print(send_to_server(send_to_server_data))

if __name__ == '__main__':
    #CreateRSAKeys()
    main()

In [ ]:
r = requests.get(end_point + "/poolList")
print(r.json())

In [ ]:
r = requests.get(end_point + "/blocks")
print(r.json())

In [ ]:
headers = {'content-type': 'application/json'}
url = end_point + "/mineBlock"
params = {}
print(requests.post(url, params=params, data=json.dumps({}), headers=headers))

In [ ]:
from IPython.display import Image, display
import os,random,string
import gzip  
import io
def gzdecode(data) :  
    compressedstream = io.BytesIO(data)  
    gziper = gzip.GzipFile(fileobj=compressedstream)    
    data2 = gziper.read()
    return data2   
    
def queryBlockByName(name):
    headers = {'content-type': 'application/json'}
    url = end_point + "/queryBlockByName?queryName=" + name
    print(requests.get(url, headers=headers))
    return requests.get(url, headers=headers).json()
search_result= queryBlockByName("sFq0wAC.jpg")
for file_binary in search_result:
    write_file_data = b64decode(bytes(file_binary, encoding = 'utf8'))
    filename = ''.join(random.choices(string.ascii_uppercase + string.digits, k = 10))
    filename += '.jpeg'
    write_file_data = gzdecode(write_file_data)
    with open(filename,'wb') as f:
        f.write(write_file_data)
    display(Image(filename = filename))
    os.remove(filename)
    print(search_result[file_binary])